####  PCA (Principal Component Analysis) is used to reduce the number of features and then compares seven Machine Learning algorithms to find which one predicts the wine categories most accurately.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score


In [2]:
dataset = pd.read_csv("Wine.csv")

X = dataset.iloc[:, 0:13].values
y = dataset.iloc[:, 13].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

In [3]:
components_list = [2, 3, 5, 8, 10, 13]

results = []

for n in components_list:
#The pipeline fits the scaler and PCA separately within each cross-validation fold. This helps prevent information from 
#the validation fold leaking into the training process.
    model = make_pipeline(
        StandardScaler(),
        PCA(n_components=n),
        SVC(kernel='rbf')
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='accuracy'
    )

    mean_accuracy = scores.mean()

    results.append({
        "PCA Components": n,
        "Mean CV Accuracy": mean_accuracy
    })

result_df = pd.DataFrame(results)

print(result_df)

   PCA Components  Mean CV Accuracy
0               2          0.957635
1               3          0.950985
2               5          0.958128
3               8          0.979064
4              10          0.985961
5              13          0.985961


In [4]:
best_row = result_df.loc[
    result_df["Mean CV Accuracy"].idxmax()
]

print("Best number of PCA components:",
      int(best_row["PCA Components"]))

print("Best mean CV accuracy:",
      round(best_row["Mean CV Accuracy"], 4))

Best number of PCA components: 10
Best mean CV accuracy: 0.986


In [5]:
best_n = int(best_row["PCA Components"])

best_model = make_pipeline(
    StandardScaler(),
    PCA(n_components=best_n),
    SVC(kernel='rbf')
)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print("Selected PCA components:", best_n)
print("Final test accuracy:", round(test_accuracy, 4))

Selected PCA components: 10
Final test accuracy: 1.0
